# Plan 4 realized-actuation challenge

This notebook reads the immutable controller-only Phase 5 artifact. Predictive outcomes remain sealed because no schedule passed the predeclared actuation gate.

In [ ]:
from pathlib import Path
import json
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

ROOT = Path.cwd().parent if Path.cwd().name == 'mnist_experiment' else Path.cwd()
ARTIFACT = ROOT / 'cache/mnist_experiment/plan4/challenge/analysis/phase5_actuation__266e6f391abb'
if not (ARTIFACT / 'COMPLETED').is_file():
    raise FileNotFoundError(f'completed Phase 5 artifact is required: {ARTIFACT}')
SUMMARY = json.loads((ARTIFACT / 'summary.json').read_text(encoding='utf-8'))
if SUMMARY['selection_uses_predictive_metrics']:
    raise RuntimeError('schedule selection unexpectedly used predictive metrics')
print({'decision': SUMMARY['decision'], 'selected_schedule': SUMMARY['selected_schedule']})

## Frozen gate

A schedule must exceed $\pi=.07$ on at least ten event transitions, span at least $.05$, correlate positively with its predictable signal, and reduce same-state Fisher risk by at least 5% relative to fixed $\pi=.05$.

In [ ]:
records = []
for schedule, schedule_result in SUMMARY['schedules'].items():
    result = schedule_result['conditions']['adaptive-fisher-h005']
    records.append({
        'schedule': schedule,
        'event transitions': result['event_transition_count'],
        'transitions above .07': result['event_signal_transition_count'],
        'event pi range': result['event_applied_pi_range'],
        'signal/action correlation': result['event_signal_action_correlation'],
        'risk reduction vs fixed .05': result['event_relative_risk_reduction_vs_fixed_005'],
        'maximum pi': result['full_applied_pi_max'],
        'passes': result['passes_actuation_gate'],
    })
gate = pd.DataFrame(records).set_index('schedule').loc[['linear', 'logistic-k32', 'logistic-k64', 'logistic-k128', 'logistic-k256']]
gate.style.format({
    'event pi range': '{:.3f}',
    'signal/action correlation': '{:.3f}',
    'risk reduction vs fixed .05': '{:.1%}',
    'maximum pi': '{:.3f}',
})

## Realized Fisher-risk actuation

The Fisher controller responds coherently to sharper shocks, but the response is not sustained long enough to pass the frozen gate.

In [ ]:
order = ['linear', 'logistic-k32', 'logistic-k64', 'logistic-k128', 'logistic-k256']
colors = dict(zip(order, ['#356a8a', '#2f855a', '#b56a22', '#7b5aa6', '#b23a48']))
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for schedule in order:
    trajectory = SUMMARY['schedules'][schedule]['conditions']['adaptive-fisher-h005']['trajectory']
    p = np.asarray([row['p'] for row in trajectory])
    pi = np.asarray([row['applied_pi'] for row in trajectory])
    axes[0].plot(p, pi, label=schedule, color=colors[schedule], linewidth=1.8)
axes[0].axhline(.05, color='black', linestyle=':', linewidth=1, label='fixed .05')
axes[0].axhline(.07, color='#555555', linestyle='--', linewidth=1, label='gate threshold')
axes[0].set(xlabel='digit-9 prevalence p', ylabel='applied pi', title='Predictable Fisher action')
axes[0].legend(fontsize=8, ncol=2)
x = np.arange(len(order))
risk = 100 * gate.loc[order, 'risk reduction vs fixed .05'].to_numpy()
axes[1].bar(x, risk, color=[colors[name] for name in order])
axes[1].axhline(5, color='#555555', linestyle='--', linewidth=1, label='5% gate')
axes[1].set_xticks(x, ['linear', 'k32', 'k64', 'k128', 'k256'])
axes[1].set(xlabel='schedule', ylabel='same-state risk reduction (%)', title='Action value under controller risk')
axes[1].legend(fontsize=8)
fig.tight_layout()

## Decision

The development screen stops here. Fisher-risk adaptation is retained as a coherent offline diagnostic, not promoted as a practical learner policy. Fixed $\pi=.05$ remains the applied baseline.

## Exploratory lower-floor sensitivity

After the gate closed, one descriptive paired replica tested fixed $\pi=.025$ and Fisher adaptive $\pi_{\min}=.025$ against the incumbent fixed $.05$. This is exploratory evidence, not confirmation.

In [ ]:
FLOOR_ARTIFACT = ROOT / 'cache/mnist_experiment/plan4/challenge/floor_analysis/floor_sensitivity__b022b149d154'
if not (FLOOR_ARTIFACT / 'COMPLETED').is_file():
    raise FileNotFoundError(f'completed floor-sensitivity artifact is required: {FLOOR_ARTIFACT}')
FLOOR = json.loads((FLOOR_ARTIFACT / 'summary.json').read_text(encoding='utf-8'))
if FLOOR['confirmatory']:
    raise RuntimeError('floor sensitivity must remain exploratory')
floor_controller = []
for schedule in order:
    value = FLOOR['schedules'][schedule]['conditions']['adaptive-fisher-pimin0025-h005']['controller']
    floor_controller.append({
        'schedule': schedule,
        'mean pi': value['full_applied_pi_mean'],
        'maximum pi': value['full_applied_pi_max'],
        'steps above .07': value['event_steps_above_007'],
        'risk reduction vs fixed .025': value['event_relative_risk_reduction_vs_fixed_0025'],
        'signal/action correlation': value['event_signal_action_correlation'],
    })
pd.DataFrame(floor_controller).set_index('schedule').style.format({
    'mean pi': '{:.3f}',
    'maximum pi': '{:.3f}',
    'risk reduction vs fixed .025': '{:.1%}',
    'signal/action correlation': '{:.3f}',
})

In [ ]:
metric_labels = {
    'environment_accuracy': 'Environmental accuracy',
    'nine_ovr_accuracy': '9 OvR accuracy',
    'nine_precision': '9 precision',
    'nine_recall': '9 recall',
}
fig, axes = plt.subplots(2, 2, figsize=(12, 7.5), sharex=True)
x = np.arange(len(order))
for axis, (field, label) in zip(axes.flat, metric_labels.items()):
    adaptive = [100 * FLOOR['schedules'][name]['adaptive_minus_fixed_0025'][field]['full_mean_difference'] for name in order]
    lower_fixed = [100 * FLOOR['schedules'][name]['fixed_0025_minus_fixed_005'][field]['full_mean_difference'] for name in order]
    axis.bar(x - .18, adaptive, width=.36, label='adaptive .025 - fixed .025', color='#b23a48')
    axis.bar(x + .18, lower_fixed, width=.36, label='fixed .025 - fixed .05', color='#356a8a')
    axis.axhline(0, color='black', linewidth=.8)
    axis.set_title(label)
    axis.set_ylabel('mean difference (percentage points)')
    axis.set_xticks(x, ['linear', 'k32', 'k64', 'k128', 'k256'])
axes[0, 0].legend(fontsize=8)
fig.suptitle('Paired predictive sensitivity across the complete trajectory')
fig.tight_layout()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.2))
for schedule in order:
    trajectory = FLOOR['schedules'][schedule]['conditions']['adaptive-fisher-pimin0025-h005']['controller']['trajectory']
    axes[0].plot([row['p'] for row in trajectory], [row['applied_pi'] for row in trajectory], color=colors[schedule], label=schedule, linewidth=1.8)
axes[0].axhline(.025, color='black', linestyle=':', linewidth=1, label='floor .025')
axes[0].set(xlabel='digit-9 prevalence p', ylabel='applied pi', title='Lower-floor adaptive actions')
axes[0].legend(fontsize=8, ncol=2)
adaptive_nll = [FLOOR['schedules'][name]['adaptive_minus_fixed_0025']['nll']['full_mean_difference'] for name in order]
fixed_nll = [FLOOR['schedules'][name]['fixed_0025_minus_fixed_005']['nll']['full_mean_difference'] for name in order]
axes[1].bar(x - .18, adaptive_nll, width=.36, label='adaptive .025 - fixed .025', color='#b23a48')
axes[1].bar(x + .18, fixed_nll, width=.36, label='fixed .025 - fixed .05', color='#356a8a')
axes[1].axhline(0, color='black', linewidth=.8)
axes[1].set_xticks(x, ['linear', 'k32', 'k64', 'k128', 'k256'])
axes[1].set(ylabel='mean NLL difference', title='Predictive likelihood; lower is better')
axes[1].legend(fontsize=8)
fig.tight_layout()

### Exploratory reading

Fixed $.025$ improves the broad predictive trajectory over fixed $.05$ in this replica. Adaptive $.025$ lowers its own estimated one-step Fisher risk but is predictively worse than fixed $.025$ across all five schedules. The discrepancy is evidence that the controller risk is not a reliable surrogate for the applied objective in this experiment.